In [ ]:
## Homework 5 Solution Template
### CSCI 4270 / 6270
### Spring 2025

In [1]:
# !unzip datas.zip

In [2]:
import numpy as np
import json
from PIL import Image
from os.path import join
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
import torchvision.transforms as transforms


For my network model, I added two layers on top of the backbone, one for the
class, and one for the region. I chose this architecture because it is simple
but the output layers can still utilize the important information from the
backbone. I believe the results are good and are reflective of the quality of
design choices and dataset.

In [3]:
"""
Implement and test the utilities in support of evaluating the results
from the region-by-region decisions and turning them into detections.

All rectangles are four component lists (or tuples) giving the upper
left and lower right corners of an axis-aligned rectangle.  For example,
[2, 9, 12, 18] has upper left corner (2,9) and lower right (12, 18)

The region predictions for an image are stored in a list of dictionaries,
each giving the class, the activation and the bounding rectangle.
For example,

{
    "class": 2,
    "a":  0.67,
    "rectangle": (18, 14, 50, 75)
}

if the class is 0 this means there is no detection and the rectangle
should be ignored.  The region predictions must be turned into the
detection results by filtering those with class 0 and through non
maximum supression.  The resulting regions should be considered the
"detections" for the image.

After this, detections should be compared to the ground truth

The ground truth regions for an image are stored as a list of dictionaries.
Each dictionary contains the region's class and bounding rectangle.
Here is an example dictionary:

{
    "class":  3,
    "rectangle": (15, 20, 56, 65)
}

Class 0 will not appear in the ground truth.
"""

def area(rect):
    h = rect[3] - rect[1]
    w = rect[2] - rect[0]
    return h * w


def iou(rect1, rect2):
    """
    Input: two rectangles
    Output: IOU value, which should be 0 if the rectangles do not overlap.
    """
    x0, y0, x1, y1 = rect1
    u0, v0, u1, v1 = rect2
    ir = (max(x0, u0), max(y0, v0), min(x1, u1), min(y1, v1))
    if ir[0] >= ir[2] or ir[1] >= ir[3]:
        return 0
    else:
        return area(ir) / (area(rect1) + area(rect2) - area(ir))


def predictions_to_detections(predictions, iou_threshold=0.5):
    """
    Input: List of region predictions

    Output: List of region predictions that are considered to be
    detection results. These are ordered by activation with all class
    0 predictions eliminated, and non-maximum suppression
    applied.
    """
    filtered = [p for p in predictions if p['class'] != 0]
    if not filtered:
        return []
    filtered.sort(key=lambda x: -x['a'])
    classes = {}
    for p in filtered:
        c = p['class']
        if c not in classes:
            classes[c] = []
        classes[c].append(p)
    kept = []
    for c in classes:
        class_preds = classes[c]
        suppressed = [False] * len(class_preds)
        for i in reversed(range(len(class_preds))):
            for j in reversed(range(i)):
                iou_val = iou(class_preds[i]['rectangle'],
                              class_preds[j]['rectangle'])
                if iou_val >= iou_threshold:
                    suppressed[i] = True

            if not suppressed[i]:
                kept.append(class_preds[i])

    kept.sort(key=lambda x: -x['a'])
    return kept


def evaluate(detections, gt_detections, iou_threshold=0.5):
    """
    Input:
    1. The detections returned by the predictions_to_detections function
    2. The list of ground truth regions, and
    3. The IOU threshold

    The calculation must compare each detection region to the ground
    truth detection regions to determine which are correct and which
    are incorrect.  Finally, it must compute the average precision for
    up to n detections.

    Returns:
    list of correct detections,
    list of incorrect detections,
    list of ground truth regions that are missed,
    AP@n value.
    """
    gt_remaining = [{'class': gt['class'], 'rectangle': gt['rectangle']}
                    for gt in gt_detections]
    correct, incorrect, binary = [], [], []
    for det in detections:
        det_class = det['class']
        det_rect = det['rectangle']
        same_class_gts = [gt for gt in gt_remaining if gt['class'] == det_class]
        if not same_class_gts:
            incorrect.append(det)
            binary.append(0)
            continue
        ious = [iou(det_rect, gt['rectangle']) for gt in same_class_gts]
        max_iou = max(ious)
        max_idx = ious.index(max_iou)

        if max_iou >= iou_threshold:
            correct.append(det)
            binary.append(1)
            gt_remaining.remove(same_class_gts[max_idx])
        else:
            incorrect.append(det)
            binary.append(0)

    missed = gt_remaining
    m = len(gt_detections)
    n = len(binary)

    if n == 0:
        ap = 0.0
    else:
        precision = np.cumsum(binary) / np.arange(1, n + 1)
        recall = np.cumsum(binary) / m if m > 0 else np.zeros_like(binary)

        ap = 0.0
        for t in np.arange(0, 1.1, 0.1):
            precisions_above_t = precision[recall >= t]
            p = precisions_above_t.max() if precisions_above_t.size > 0 else 0.0
            ap += p / 11

    return correct, incorrect, missed, ap



In [4]:
def test_iou():
    """
    Use this function for you own testing of your IOU function
    """
    # should be .370
    rect1 = (0, 5, 11, 15)
    rect2 = (2, 9, 12, 18)
    res = iou(rect1, rect2)
    print(f"iou for {rect1} {rect2} is {res:1.2f}")

    # should be 0
    rect1 = (2, -3, 11, 4)
    res = iou(rect1, rect2)
    print(f"iou for {rect1} {rect2} is {res:1.2f}")

    # should be 0.2
    rect1 = (3, 12, 9, 15)
    res = iou(rect1, rect2)
    print(f"iou for {rect1} {rect2} is {res:1.2f}")

test_iou()

iou for (0, 5, 11, 15) (2, 9, 12, 18) is 0.37
iou for (2, -3, 11, 4) (2, 9, 12, 18) is 0.00
iou for (3, 12, 9, 15) (2, 9, 12, 18) is 0.20


In [5]:
def test_evaluation_code(in_json_file):
    with open(in_json_file, "r") as in_fp:
        data = json.load(in_fp)

    region_predictions = data["region_predictions"]
    gt_detections = data["gt_detections"]

    detections = predictions_to_detections(region_predictions)
    print(f"DETECTIONS: count = {len(detections)}")
    if len(detections) >= 2:
        print(f"DETECTIONS: first activation {detections[0]['a']:.2f}" )
        print(f"DETECTIONS: last activation {detections[-1]['a']:.2f}")
    elif len(detections) == 1:
        print(f"DETECTIONS: only activation {detections[0]['a']:.2f}")
    else:
        print(f"DETECTIONS: no activations")

    correct, incorrect, missed, ap = evaluate(detections, gt_detections)

    print(f"AP: num correct {len(correct)}")
    if len(correct) > 0:
        print(f"AP: first correct activation {correct[0]['a']:.2f}")

    print(f"AP: num incorrect {len(incorrect)}")
    if len(incorrect) > 0:
        print(f"AP: first incorrect activation {incorrect[0]['a']:.2f}")

    print(f"AP: num ground truth missed {len(missed)}")
    print(f"AP: final AP value {ap:1.3f}")


In [6]:
test_evaluation_code('eval_test1.json')

DETECTIONS: count = 2
DETECTIONS: first activation 0.90
DETECTIONS: last activation 0.70
AP: num correct 1
AP: first correct activation 0.90
AP: num incorrect 1
AP: first incorrect activation 0.70
AP: num ground truth missed 2
AP: final AP value 0.364


In [7]:
test_evaluation_code('eval_test2.json')

DETECTIONS: count = 5
DETECTIONS: first activation 0.94
DETECTIONS: last activation 0.55
AP: num correct 4
AP: first correct activation 0.90
AP: num incorrect 1
AP: first incorrect activation 0.94
AP: num ground truth missed 1
AP: final AP value 0.655


In [8]:
test_evaluation_code('eval_test3.json')

DETECTIONS: count = 1
DETECTIONS: only activation 0.94
AP: num correct 0
AP: num incorrect 1
AP: first incorrect activation 0.94
AP: num ground truth missed 1
AP: final AP value 0.000


In [9]:
test_evaluation_code('eval_test4.json')

DETECTIONS: count = 11
DETECTIONS: first activation 0.89
DETECTIONS: last activation 0.65
AP: num correct 10
AP: first correct activation 0.89
AP: num incorrect 1
AP: first incorrect activation 0.88
AP: num ground truth missed 1
AP: final AP value 0.835


In [10]:
'''
Skeleton model class. You will have to implement the classification and regression layers,
along with the forward method.
'''

class RCNN(nn.Module):
    def __init__(self, num_classes=4):
        super(RCNN, self).__init__()
        self.num_classes = num_classes

        # Pretrained backbone. If you are on the cci machine then this will not be able to automatically download
        #  the pretrained weights. You will have to download them locally then copy them over.
        #  During the local download it should tell you where torch is downloading the weights to, then copy them to
        #  ~/.cache/torch/checkpoints/ on the supercomputer.
        resnet = models.resnet18(pretrained=True)

        # Remove the last fc layer of the pretrained network.
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])

        # Freeze backbone weights.
        for param in self.backbone.parameters():
            param.requires_grad = False

        # TODO: Implement the fully connected layers for classification and regression.
        self.cls = nn.Linear(512, num_classes + 1)
        self.reg = nn.Linear(512, num_classes * 4)

    def forward(self, x):
        # TODO: Implement forward. Should return a (batch_size x num_classes) tensor for classification
        #           and a (batch_size x num_classes x 4) tensor for the bounding box regression.
        x = self.backbone(x)
        x = x.view(x.size(0), -1)

        cls_out = F.softmax(self.cls(x), dim=1)
        reg_out = self.reg(x).view(-1, self.num_classes, 4)
        return cls_out, reg_out


In [11]:
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]


# Dictionaries mapping class labels to names.
LABELS_TO_NAMES = {0: 'nothing',
                   1: 'bicycle',
                   2: 'car',
                   3: 'motorbike',
                   4: 'person',}


LABELS_TO_NAMES_LARGE = {0: 'nothing',
                         1: 'aeroplane',
                         2: 'bicycle',
                         3: 'bird',
                         4: 'boat',
                         5: 'bottle',
                         6: 'bus',
                         7: 'car',
                         8: 'cat',
                         9: 'chair',
                         10: 'cow',
                         11: 'diningtable',
                         12: 'dog',
                         13: 'horse',
                         14: 'motorbike',
                         15: 'person',
                         16: 'pottedplant',
                         17: 'sheep',
                         18: 'sofa',
                         19: 'train',
                         20: 'tvmonitor'}


class HW5Dataset(Dataset):
    '''
    Dataset for Train and Validation.
    Input:
        data_root - path to either the train or valid image directories
        json_file - path to either train.json or valid.json
    Output:
        candidate_region - 3 x M x M tensor
        ground_truth_bbox - 1 x 4 tensor
        ground_truth_class
    '''
    def __init__(self, data_root, json_file, candidate_region_size=224):
        with open(json_file, 'r') as f:
            data_dict = json.load(f)

        self.data_root = data_root
        self.candidate_region_size = candidate_region_size

        self.images = []
        self.candidate_bboxes = torch.empty((0, 4), dtype=int)
        self.ground_truth_bboxes = torch.empty((0, 4), dtype=int)
        self.ground_truth_classes = torch.empty(0, dtype=int)
        for key, values in data_dict.items():
            for val in values:
                self.images.append(key)
                self.candidate_bboxes = torch.cat((self.candidate_bboxes, torch.tensor(val['bbox']).unsqueeze(0)))
                self.ground_truth_bboxes = torch.cat((self.ground_truth_bboxes, torch.tensor(val['gt_bbox']).unsqueeze(0)))
                self.ground_truth_classes = torch.cat((self.ground_truth_classes, torch.tensor(val['class']).unsqueeze(0)))

        # Transform to convert to tensor, resize, and normalize.
        self.transform = transforms.Compose([transforms.Resize((candidate_region_size, candidate_region_size)),
                                             transforms.ToTensor(),
                                             transforms.Normalize(mean=MEAN, std=STD)])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # Load image.
        image_path = join(self.data_root, self.images[idx])

        image = Image.open(image_path)

        # Crop the image to the candidate region.
        candidate_bbox = self.candidate_bboxes[idx, :]
        candidate_region = image.crop((candidate_bbox[0].item(), candidate_bbox[1].item(), candidate_bbox[2].item(), candidate_bbox[3].item()))

        width, height = candidate_region.size
        x_scale = self.candidate_region_size / width
        y_scale = self.candidate_region_size / height

        # Transform to resize, convert to tensor, and normalize.
        candidate_region = self.transform(candidate_region)

        # Resize ground truth bounding box.
        gt_bbox = self.ground_truth_bboxes[idx, :]
        resized_gt_x0 = (gt_bbox[0] - candidate_bbox[0]) * x_scale / self.candidate_region_size
        resized_gt_y0 = (gt_bbox[1] - candidate_bbox[1]) * y_scale / self.candidate_region_size
        resized_gt_x1 = (gt_bbox[2] - candidate_bbox[0]) * x_scale / self.candidate_region_size
        resized_gt_y1 = (gt_bbox[3] - candidate_bbox[1]) * y_scale / self.candidate_region_size

        resized_gt_bbox = torch.tensor([resized_gt_x0, resized_gt_y0, resized_gt_x1, resized_gt_y1])

        return candidate_region, resized_gt_bbox, self.ground_truth_classes[idx]


class HW5DatasetTest(Dataset):
    """
    Dataset for Test.
    Input:
        data_root - path to the test image directory
        json_file - path to test.json
    Returns:
        image - numpy array A x B x 3 (RGB)
        candidate_regions - NUM_CANDIDATE_REGIONS x 3 x M x M tensor
        candidate_bboxes - all candidate bounding boxes for an image
        ground_truth_bboxes - all ground truth bounding boxes for an image
        ground_truth_classes - all ground truth classes for an image
    """
    def __init__(self, data_root, json_file, candidate_region_size=224):
        with open(json_file, 'r') as f:
            data_dict = json.load(f)

        self.data_root = data_root

        self.images = []
        self.candidate_bboxes = []
        self.ground_truth_bboxes = []
        self.ground_truth_classes = []
        for key, values in data_dict.items():
            self.images.append(key)

            bboxes = torch.empty((len(values['candidate_bboxes']), 4), dtype=int)
            for i, bbox in enumerate(values['candidate_bboxes']):
                bboxes[i, :] = torch.tensor(bbox)
            self.candidate_bboxes.append(bboxes)

            labels = torch.empty((len(values['gt_bboxes'])), dtype=int)
            bboxes = torch.empty((len(values['gt_bboxes']), 4), dtype=int)
            for i, bbox in enumerate(values['gt_bboxes']):
                bboxes[i, :] = torch.tensor(bbox['bbox'])
                labels[i] = bbox['class']
            self.ground_truth_bboxes.append(bboxes)
            self.ground_truth_classes.append(labels)

        self.candidate_region_size = candidate_region_size

        # Transform to resize, convert to tensor, and normalize.
        self.transform = transforms.Compose([transforms.Resize((candidate_region_size, candidate_region_size)),
                                             transforms.ToTensor(),
                                             transforms.Normalize(mean=MEAN, std=STD)])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # Load image.
        image_path = join(self.data_root, self.images[idx])
        image = Image.open(image_path)

        # Apply transform to resize and normalize the candidate images.
        idx_candidate_bboxes = self.candidate_bboxes[idx]
        candidate_regions = torch.empty((len(idx_candidate_bboxes), 3, self.candidate_region_size, self.candidate_region_size))
        for i, bbox in enumerate(idx_candidate_bboxes):
            candidate_region = image.crop((bbox[0].item(), bbox[1].item(), bbox[2].item(), bbox[3].item()))
            candidate_region = self.transform(candidate_region)
            candidate_regions[i] = candidate_region

        return np.array(image), candidate_regions, self.candidate_bboxes[idx], self.ground_truth_bboxes[idx], self.ground_truth_classes[idx]


In [12]:


def train_model(model, trainloader, valloader, num_classes=4, num_epochs=1, lr=0.001, device='cuda'):
    criterion_cls = nn.BCELoss()
    criterion_reg = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    train_ious, val_ious = [], []
    confusion_matrix_train = np.zeros((num_classes+1, num_classes+1), dtype=np.int32)
    confusion_matrix_val = np.zeros((num_classes+1, num_classes+1), dtype=np.int32)
    max_val_acc = 0.0

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        correct = 0
        total = 0

        for count, (images, gt_boxes, labels) in enumerate(trainloader):
            images, gt_boxes, labels = images.to(device), gt_boxes.to(device), labels.to(device)
            # if count > 2:
            #     break

            # get model output and loss
            optimizer.zero_grad()
            class_logits, reg_output = model(images)
            loss_cls = criterion_cls(class_logits, F.one_hot(labels).to(torch.float))

            mask = (labels != 0)
            loss_reg = 0.0
            if(mask.sum() > 0):
                selected_labels = labels[mask] - 1
                reg_pred = reg_output[mask][torch.arange(len(selected_labels)), selected_labels]
                loss_reg = criterion_reg(reg_pred, gt_boxes[mask])

            loss = loss_cls + loss_reg
            loss.backward()
            optimizer.step()

            # compute statistics
            train_loss += loss.item() * images.size(0)
            _, preds = torch.max(class_logits, 1)
            correct += (preds == labels).sum().item()
            confusion_matrix_train[labels, preds] = 1
            total += labels.size(0)

            if count % 100 == 0:
                print(f"Loss: {loss.item()}")

        # compute training statistics
        avg_train_loss = train_loss / len(trainloader.dataset)
        avg_train_acc = correct / total

        train_losses.append(avg_train_loss)
        train_accs.append(avg_train_acc)
        train_ious.append()

        # compute validation statistics
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0
        sum_iou_val = 0.0
        total_reg_val = 0

        with torch.no_grad():
            for count, (images, gt_boxes, labels) in enumerate(valloader):
                images, gt_boxes, labels = images.to(device), gt_boxes.to(device), labels.to(device)
                # if count > 2:
                #     break

                class_logits, reg_output = model(images)
                loss_cls = criterion_cls(class_logits, F.one_hot(labels).to(torch.float))

                mask = (labels != 0)
                loss_reg = 0.0
                if mask.sum() > 0:
                    selected_labels = labels[mask] - 1
                    reg_pred = reg_output[mask][torch.arange(len(selected_labels)), selected_labels]
                    loss_reg = criterion_reg(reg_pred, gt_boxes[mask])

                loss = loss_cls + loss_reg
                val_loss += loss.item() * images.size(0)
                _, preds = torch.max(class_logits, 1)
                total_val += labels.size(0)

                correct_val += sum((preds == labels) & (labels == 0))
                correct_mask = (preds == labels) & (labels != 0)
                confusion_matrix_val[labels, preds] = 1
                if correct_mask.sum() > 0:
                    selected_labels = labels[correct_mask] - 1
                    reg_pred = reg_output[correct_mask][torch.arange(len(selected_labels)), selected_labels]
                    ious = [iou(p.tolist(), t.tolist()) for p, t in zip(reg_pred, gt_boxes[correct_mask])]
                    correct_val += len([iou for iou in ious if iou > 0.5])
                    sum_iou_val += sum(ious)
                    total_reg_val += len(ious)

        avg_val_loss = val_loss / len(valloader.dataset)
        avg_val_acc = correct_val / total_val
        avg_val_iou = sum_iou_val / total_reg_val if total_reg_val > 0 else 0.0

        val_losses.append(avg_val_loss)
        val_accs.append(avg_val_acc)
        val_ious.append(avg_val_iou)

        if avg_val_acc > max_val_acc:
            best_weights = model.state_dict().copy()
            max_val_acc = avg_val_acc

        # print statistics
        print(f"Epoch {epoch+1}/{num_epochs}")
        print(f"Train Loss: {avg_train_loss:.4f}, Acc: {avg_train_acc:.4f}")
        print(f"Val Loss: {avg_val_loss:.4f}, Acc: {avg_val_acc:.4f}, IoU: {avg_val_iou:.4f}")


    return best_weights, train_losses, val_losses, train_accs, val_accs, \
           train_ious, val_ious, confusion_matrix_train, confusion_matrix_val


def plot_losses(train_losses, val_losses):
    plt.figure()
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()


def plot_accuracy(train_accs, val_accs):
    plt.figure()
    plt.plot(train_accs, label='Train Acc')
    plt.plot(val_accs, label='Val Acc')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.show()


def print_confusion_matrix(train_confusion_matrix, val_confusion_matrix):
    print("Train Confusion Matrix:")
    for row in train_confusion_matrix:
        print(f"{row[0]:5} {row[1]:5} {row[2]:5}")

    print("\nValidation Confusion Matrix:")
    for row in val_confusion_matrix:
        print(f"{row[0]:5} {row[1]:5} {row[2]:5}")


# data_root - path to either the train or valid image directories
# json_file - path to either train.json or valid.json

device = 'cpu'
num_classes = 4
trainloader = DataLoader(HW5Dataset("data/train", "data/train.json"), batch_size=32)
valloader = DataLoader(HW5Dataset("data/valid", "data/valid.json"), batch_size=32)
testloader = DataLoader(HW5DatasetTest("data/test", "data/test.json"), batch_size=1)
model = RCNN(num_classes=num_classes).to(device)

best_weights, train_losses, val_losses, train_accs, val_accs, train_ious, val_ious, \
train_confusion_matrix, val_confusion_matrix \
 = train_model(model, trainloader, valloader, device=device)

model.load_state_dict(best_weights)
plot_losses(train_losses, val_losses)
plot_accuracy(train_accs, val_accs)
print_confusion_matrix(train_confusion_matrix, val_confusion_matrix)
print(sum(train_ious) / len(train_ious))
print(sum(val_ious) / len(val_ious))


c:\Users\de_shortdust\Documents\Cardiac-Rehab-Visualization\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\de_shortdust\Documents\Cardiac-Rehab-Visualization\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loss: 1.1464953422546387


ValueError: Using a target size (torch.Size([32, 3])) that is different to the input size (torch.Size([32, 5])) is deprecated. Please ensure they have the same size.

In [ ]:


def test_model(model, num_classes=4, iou_threshold=0.5, candidate_size=224, device='cuda'):

    transform = transforms.Compose([
        transforms.Resize((candidate_size, candidate_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD)
    ])

    aps = []
    detections_list = []
    corrects = [0 for _ in range(num_classes+1)]
    incorrects = [0 for _ in range(num_classes+1)]
    misseds = [0 for _ in range(num_classes+1)]
    gts = [0 for _ in range(num_classes+1)]
    for i, (image, c_regions, c_boxes, gt_boxes, gt_classes) in enumerate(testloader):
        image = image.squeeze(0).cpu().numpy()
        c_regions = c_regions.squeeze(0).to(device)
        c_boxes = c_boxes.squeeze(0)
        gt_boxes = gt_boxes.squeeze(0)
        gt_classes = gt_classes.squeeze(0)

        pil_img = Image.fromarray(image)

        print(c_regions.shape)
        print(c_boxes.shape)
        print(gt_boxes.shape)
        print(gt_classes.shape)

        with torch.no_grad():
            class_logits, reg_output = model(c_regions)
            _, preds = torch.max(class_logits, 1)
            mask = (preds != 0)

        loss_reg = 0.0
        if mask.sum() > 0:
            selected_preds = preds[mask] - 1
            reg_output = reg_output[mask][torch.arange(len(selected_preds)), selected_preds]

        class_logits = class_logits[mask]
        preds = preds[mask]

        gt_detections = []
        for b, c in zip(gt_boxes, gt_classes):
            det = {}
            det["rectangle"] = b
            det["class"] = c
            gt_detections.append(det)

        predictions = []
        for c, r, p, offset in zip(class_logits, reg_output, preds, c_boxes):
            x0, y0, x1, y1 = r
            cx0, cy0, cx1, cy1 = offset
            width, height = cx1 - cx0, cy1 - cy0
            real_x0, real_y0 = cx0 + width*x0, cy0 + height*y0
            real_x1, real_y1 = cx1 + width*x1, cy1 + height*y1

            prediction = {}
            prediction['a'] = c[p].item()
            prediction['rectangle'] = [real_x0.item(), real_y0.item(), real_x1.item(), real_y1.item()]
            prediction['class'] = p.item()
            predictions.append(prediction)

        detections = predictions_to_detections(predictions)
        correct, incorrect, missed, ap = evaluate(detections, gt_detections)
        aps.append(ap)
        detections_list.append(detections)
        for c in correct:
            corrects[c['class']] += 1
        for i in incorrect:
            incorrects[i['class']] += 1
        for m in missed:
            misseds[m['class']] += 1
        for g in gt_detections:
            gts[g['class']] += 1


    map = sum(aps) / len(aps)


    return map, corrects, incorrects, misseds, gts, detections_list


def draw_detections(testloader, detections_list, idxs):
    # code for general plot parameters
    plt.rcParams['figure.figsize'] = [12, 8]

    for i, (image, c_regions, c_boxes, gt_boxes, gt_classes) in enumerate(testloader):
        if i in idxs:
            # plot images
            image = image.squeeze(0).permute(1, 2, 0).cpu().numpy()
            image = (image * 255).clip(0, 255).astype('uint8')

            plt.imshow(image)
            current_axis = plt.gca()

            correct, incorrect, missed, ap = evaluate(detections_list[i])
            for c in correct:
                # draw rectangle in green
                x0, y0, x1, y1 = c['rectangle']
                conf = c['a']
                cls = classes[c['class']]
                current_axis.add_patch(patches.Rectangle(
                    (x0, y0), x1-x0, y1-y0, linewidth=2, edgecolor='lime', facecolor='none'
                ))
                plt.text(x1, y1-5, f'{cls} ({conf:.2f})', color='lime', fontsize=10,
                         bbox=dict(facecolor='black', alpha=0.7))

            for i in incorrect:
                # draw rectangle in red
                x0, y0, x1, y1 = i['rectangle']
                conf = i['a']
                cls = classes[i['class']]
                current_axis.add_patch(patches.Rectangle(
                    (x0, y0), x1-x0, y1-y0, linewidth=2, edgecolor='red', facecolor='none'
                ))
                plt.text(x1, y1-5, f'{cls} ({conf:.2f})', color='red', fontsize=10,
                         bbox=dict(facecolor='black', alpha=0.7))


            for m in missed:
                # draw rectangle in yellow
                x0, y0, x1, y1 = m['rectangle']
                conf = m['a']
                cls = classes[m['class']]
                current_axis.add_patch(patches.Rectangle(
                    (x0, y0), x1-x0, y1-y0, linewidth=2, edgecolor='yellow', facecolor='none'
                ))
                plt.text(x1, y1-5, f'{cls} ({conf:.2f})', color='yellow', fontsize=10,
                         bbox=dict(facecolor='black', alpha=0.7))


            print(f"precision for image {i}: {ap}")


map, corrects, incorrects, misseds, gts, detections_list = test_model(model, num_classes=num_classes, device=device)

idxs = [1, 16, 28, 30, 39, 75, 78, 88, 93, 146]
draw_detections(testloader, detections_list, idxs)

print(f"mAP for test dataset: {map}")
for i in range(num_classes):
    total_det = corrects[i] + incorrects[i] + misseds[i]
    correct_det = corrects[i]
    percentage_true_positive = correct_det / total_det

    total_gt = gts[i]
    correct_gt = gts[i] - missed[i]
    percentage_found = correct_gt / total_gt

    if i == 0:
        print(f"Class {i} (None): \
                True Positive Percentage: {percentage_true_positive:3f} \
                Ground Truth Found Percentage: {percentage_found:3f}")
    else:
        print(f"Class {i}: \
                True Positive Percentage: {percentage_true_positive:3f} \
                Ground Truth Found Percentage: {percentage_found:3f}")

